# Notebook 06 – Kiểm định mô hình

Notebook này **không tạo đặc trưng mới**; chỉ kiểm định các mô hình đã xây ở notebook 03 và 05 theo các tiêu chí học thuật và ứng dụng:

1. **Độ chính xác có ý nghĩa thống kê**: khoảng tin cậy bootstrap 95% của AUC/Gini; kiểm định DeLong cho chênh lệch AUC giữa các mô hình.
2. **Độ ổn định theo thời gian**: chỉ số ổn định Gini của cuộc thi (trung bình Gini tuần + phạt xu hướng giảm + phạt dao động).
3. **Kiểm định cửa sổ thời gian** (rolling-origin): cửa sổ mở rộng và cửa sổ trượt, 4 khối kiểm định liên tiếp, có khoảng đệm.
4. **Hiệu chỉnh xác suất**: ECE, hệ số độ dốc / chặn của hiệu chỉnh trước và sau isotonic.
5. **Trôi dữ liệu**: adversarial validation (phân biệt train với test) và PSI từng biến.
6. **Kiểm tra rò rỉ của quy trình**: huấn luyện với nhãn bị hoán vị ngẫu nhiên – AUC phải ≈ 0,5.

Tập test (tuần 78–91) chỉ dùng để báo cáo, không dùng để chọn tham số.

In [ ]:
# Giai phong bo nho con lai tu cac phan truoc (chay TRUOC khi doc du lieu cua phan nay)
import utils
utils.free_memory(globals())

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, polars as pl
import matplotlib.pyplot as plt
import utils
from utils import META
import lightgbm as lgb
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score

DATA_DIR, OUT_DIR = utils.get_data_dir(), utils.get_out_dir()
SEED = 42
summary = {}          # bang tong hop cuoi notebook

pv = pd.read_parquet(OUT_DIR / "preds_all_valid.parquet")
pt = pd.read_parquet(OUT_DIR / "preds_all_test.parquet")
MODELS = [c for c in pt.columns if c not in ("case_id", "WEEK_NUM", "target")]
print("Mo hinh co du doan:", MODELS)

## 1. Khoảng tin cậy bootstrap và kiểm định DeLong (tập test)

In [ ]:
rows = []
for m in MODELS:
    auc, lo, hi = utils.bootstrap_ci(pt["target"], pt[m], n=500, seed=SEED)
    rows.append({"mo_hinh": m, "AUC": auc, "AUC_CI95_thap": lo, "AUC_CI95_cao": hi,
                 "Gini": 2 * auc - 1, "Gini_CI95_thap": 2 * lo - 1, "Gini_CI95_cao": 2 * hi - 1})
ci = pd.DataFrame(rows).set_index("mo_hinh").sort_values("AUC", ascending=False)
ci.to_csv(OUT_DIR / "auc_bootstrap_ci.csv"); display(ci.round(4))

In [ ]:
best = ci.index[0]
dl = []
for m in MODELS:
    if m == best: continue
    r = utils.delong_test(pt["target"], pt[best], pt[m]); r["so_sanh"] = f"{best}  vs  {m}"; dl.append(r)
dl = pd.DataFrame(dl).set_index("so_sanh").sort_values("chenh")
dl["co_y_nghia_5%"] = dl["p_value"] < 0.05
dl.to_csv(OUT_DIR / "delong_tests.csv"); display(dl.round(5))
summary["Mo hinh tot nhat (test)"] = best
summary["AUC test [CI95]"] = "%.4f [%.4f; %.4f]" % tuple(ci.loc[best, ["AUC", "AUC_CI95_thap", "AUC_CI95_cao"]])

## 2. Chỉ số ổn định Gini theo tuần (định nghĩa của cuộc thi)
`stability = mean(Gini tuần) + 88 × min(0, độ dốc) − 0,5 × std(phần dư)` – phạt mô hình có Gini giảm dần theo thời gian hoặc dao động mạnh giữa các tuần.

In [ ]:
st = pd.DataFrame({m: utils.gini_stability(pt.assign(score=pt[m])) for m in MODELS}).T.sort_values("stability", ascending=False)
st.to_csv(OUT_DIR / "gini_stability.csv"); display(st.round(4))
summary["Gini stability (test)"] = round(st.loc[best, "stability"], 4)

gw = utils.gini_by_week(pt.assign(score=pt[best]))
ax = gw.plot(marker="o", figsize=(8, 3), title=f"Gini theo tuan (test) - {best}")
x = np.arange(len(gw)); a, b = np.polyfit(x, gw.values, 1); ax.plot(gw.index, a * x + b, "--", c="gray"); plt.show()

## 3. Kiểm định cửa sổ thời gian (rolling-origin)
Bốn khối kiểm định liên tiếp, mỗi khối huấn luyện chỉ trên dữ liệu **trước** khối đó, chừa **khoảng đệm 2 tuần**:
- **Cửa sổ mở rộng**: train từ tuần 0 → trước khoảng đệm.
- **Cửa sổ trượt**: chỉ `SLIDE` tuần gần nhất.

Early stopping dùng 8 tuần cuối của chính cửa sổ huấn luyện (không chạm khối kiểm định). Để giới hạn thời gian chạy, notebook dùng `learning_rate = 0.08` nên AUC có thể thấp hơn mô hình chính đôi chút; mục đích là **so sánh giữa các khối**, không phải thay mô hình chính.

In [ ]:
import joblib
HAS_FINAL = (OUT_DIR / "final_model.joblib").exists()
if HAS_FINAL:                                    # mo hinh cuoi: dac trung goc + credit bureau (notebook 05)
    fpack = joblib.load(OUT_DIR / "final_model.joblib"); feats = fpack["features"]
    df = pl.read_parquet(OUT_DIR / "features_v2.parquet").to_pandas()
else:
    df = pl.read_parquet(OUT_DIR / "features_v1.parquet").to_pandas(); feats = [c for c in df.columns if c not in META]
print("Bo dac trung kiem dinh:", "features_v2 (mo hinh cuoi)" if HAS_FINAL else "features_v1", "|", len(feats), "bien")
BLOCKS = [(40, 52), (52, 64), (64, 78), (78, 92)]
GAP, SLIDE, ES_WEEKS = 2, 40, 8
P_FAST = dict(n_estimators=2000, learning_rate=0.08, num_leaves=31, min_child_samples=200, subsample=0.8, subsample_freq=1,
              colsample_bytree=0.6, reg_lambda=5.0, random_state=SEED, n_jobs=-1, verbose=-1, metric="auc")

def fit_eval(w_start, w_end, block):
    tr = df[(df.WEEK_NUM >= w_start) & (df.WEEK_NUM < w_end - ES_WEEKS)]
    es = df[(df.WEEK_NUM >= w_end - ES_WEEKS) & (df.WEEK_NUM < w_end)]
    te = df[(df.WEEK_NUM >= block[0]) & (df.WEEK_NUM < block[1])]
    m = lgb.LGBMClassifier(**P_FAST).fit(tr[feats], tr["target"], eval_set=[(es[feats], es["target"])],
                                         callbacks=[lgb.early_stopping(100, verbose=False)])
    p = m.predict_proba(te[feats])[:, 1]
    return {"tuan_train": f"{w_start}-{w_end - 1}", "so_ho_so_train": len(tr) + len(es), "tuan_kiem_dinh": f"{block[0]}-{block[1] - 1}",
            "AUC": roc_auc_score(te["target"], p), "Gini_tuan_tb": utils.gini_by_week(te.assign(score=p)).mean(),
            "ty_le_vo_no": te["target"].mean(), "so_cay": m.best_iteration_}

roll = []
for blk in BLOCKS:
    end = blk[0] - GAP
    roll.append({"cua_so": "mo rong", **fit_eval(0, end, blk)})
    roll.append({"cua_so": f"truot {SLIDE} tuan", **fit_eval(max(0, end - SLIDE), end, blk)})
    print("Xong khoi", blk)
roll = pd.DataFrame(roll); roll.to_csv(OUT_DIR / "rolling_origin.csv", index=False)
display(roll.round(4))
agg = roll.groupby("cua_so")["AUC"].agg(["mean", "std", "min"]); display(agg.round(4))
summary["Rolling AUC (mo rong) tb +- std"] = "%.4f +- %.4f" % tuple(agg.loc["mo rong", ["mean", "std"]])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for w, g in roll.groupby("cua_so"): ax.plot(g["tuan_kiem_dinh"], g["AUC"], "o-", label=w)
ax.set_ylabel("AUC"); ax.set_xlabel("Khoi tuan kiem dinh"); ax.set_title("Kiem dinh cua so thoi gian"); ax.legend(); plt.show()

## 4. Hiệu chỉnh xác suất (mô hình chọn ở notebook 03)
Mô hình dùng để ra quyết định hạn mức cần **xác suất đúng mức**, không chỉ xếp hạng tốt. Isotonic được fit trên valid, đánh giá trên test.
Hiệu chỉnh tốt: ECE gần 0, hệ số độ dốc gần 1, hệ số chặn gần 0.

In [ ]:
tag = "final" if HAS_FINAL else ""
pr_v = pd.read_parquet(OUT_DIR / f"preds{'_' + tag if tag else ''}_valid.parquet")
pr_t = pd.read_parquet(OUT_DIR / f"preds{'_' + tag if tag else ''}_test.parquet")
iso = IsotonicRegression(y_min=1e-4, y_max=1 - 1e-4, out_of_bounds="clip").fit(pr_v["pd_raw"], pr_v["target"])
cal = pd.DataFrame({"truoc hieu chinh": utils.calibration_stats(pr_t["target"], pr_t["pd_raw"]),
                    "sau isotonic": utils.calibration_stats(pr_t["target"], iso.predict(pr_t["pd_raw"]))}).T
cal.to_csv(OUT_DIR / "calibration_stats.csv"); display(cal.round(4))
summary["ECE sau hieu chinh"] = round(cal.loc["sau isotonic", "ECE"], 5)

# Hang rui ro voi mo hinh cuoi (diem cat xac dinh tren valid, giong notebook 04)
pr_v["pd"], pr_t["pd"] = iso.predict(pr_v["pd_raw"]), iso.predict(pr_t["pd_raw"])
FACTOR = 20 / np.log(2); OFFSET = 600 - FACTOR * np.log(50)
for d in (pr_v, pr_t): d["score"] = OFFSET + FACTOR * np.log((1 - d["pd"]) / d["pd"])
edges = np.concatenate([[-np.inf], np.unique(np.quantile(pr_v["score"], [0.10, 0.30, 0.60, 0.85])), [np.inf]])
labels = ["E", "D", "C", "B", "A"][-(len(edges) - 1):]
pr_t["band"] = pd.cut(pr_t["score"], edges, labels=labels)
bt = pr_t.groupby("band", observed=True).agg(so_ho_so=("target", "size"), ty_le_vo_no=("target", "mean"), pd_tb=("pd", "mean"))
bt["ty_trong"] = bt["so_ho_so"] / len(pr_t); bt.to_csv(OUT_DIR / "final_bands.csv")
print("Ty le vo no giam don dieu E -> A:", bool(bt["ty_le_vo_no"].is_monotonic_decreasing)); display(bt.round(4))

## 5. Trôi dữ liệu: adversarial validation và PSI từng biến
Huấn luyện một mô hình phân biệt hồ sơ **train (tuần 0–63)** với hồ sơ **test (tuần 78–91)**. AUC ≈ 0,5: hai giai đoạn giống nhau; AUC cao: có biến thay đổi phân phối theo thời gian – cần theo dõi khi triển khai.

In [ ]:
tr_w = df[df.WEEK_NUM <= 63].sample(150_000, random_state=SEED) if (df.WEEK_NUM <= 63).sum() > 150_000 else df[df.WEEK_NUM <= 63]
te_w = df[df.WEEK_NUM >= 78].sample(min(150_000, (df.WEEK_NUM >= 78).sum()), random_state=SEED)
adv = pd.concat([tr_w.assign(is_test=0), te_w.assign(is_test=1)]).sample(frac=1, random_state=SEED)
cut = int(len(adv) * 0.7)
am = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, colsample_bytree=0.6, verbose=-1, random_state=SEED)
am.fit(adv[feats].iloc[:cut], adv["is_test"].iloc[:cut])
adv_auc = roc_auc_score(adv["is_test"].iloc[cut:], am.predict_proba(adv[feats].iloc[cut:])[:, 1])
print("AUC adversarial (train vs test): %.4f" % adv_auc)
summary["AUC adversarial"] = round(adv_auc, 4)

num = [c for c in feats if pd.api.types.is_numeric_dtype(df[c])]
psi = pd.Series({c: utils.psi(tr_w[c].dropna(), te_w[c].dropna()) for c in num if tr_w[c].notna().sum() > 100 and te_w[c].notna().sum() > 100})
imp_adv = pd.Series(am.booster_.feature_importance("gain"), index=feats)
drift = pd.DataFrame({"PSI": psi, "gain_adversarial_%": 100 * imp_adv / imp_adv.sum()}).sort_values("gain_adversarial_%", ascending=False)
drift.to_csv(OUT_DIR / "feature_drift.csv")
print("So bien PSI > 0.25 (troi manh):", int((psi > 0.25).sum()), "| 0.1-0.25 (can theo doi):", int(((psi > 0.1) & (psi <= 0.25)).sum()))
display(drift.head(15).round(4))

## 6. Kiểm tra rò rỉ của quy trình: hoán vị nhãn
Nếu quy trình không rò rỉ thông tin nhãn, mô hình huấn luyện trên **nhãn bị xáo trộn ngẫu nhiên** phải cho AUC test ≈ 0,5.

In [ ]:
tr_p = df[df.WEEK_NUM <= 63].sample(min(300_000, (df.WEEK_NUM <= 63).sum()), random_state=SEED)
y_perm = np.random.default_rng(SEED).permutation(tr_p["target"].values)
te_p = df[df.WEEK_NUM >= 78]
pm = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, colsample_bytree=0.6, verbose=-1, random_state=SEED)
pm.fit(tr_p[feats], y_perm)
perm_auc = roc_auc_score(te_p["target"], pm.predict_proba(te_p[feats])[:, 1])
print("AUC test voi nhan hoan vi: %.4f (ky vong ~0.5)" % perm_auc)
summary["AUC nhan hoan vi"] = round(perm_auc, 4)

## 6b. Phân tích độ nhạy: bỏ thuộc tính nhạy cảm (giới tính)
Mô hình chính có dùng giới tính khách hàng. Trong thực tế, nhiều quy định cho vay hạn chế dùng thuộc tính nhạy cảm để ra quyết định.
Ô dưới huấn luyện lại mô hình cuối **sau khi bỏ** các cột có mô tả về giới tính (không thêm biến nào), so sánh AUC bằng DeLong và so sánh tỷ lệ duyệt theo giới tính ở cùng ngưỡng.

In [ ]:
import re
defs = utils.load_definitions()
sens = [c for c in feats if re.search(r"\bgender\b|\bsex\b", defs.get(utils.base_name(c), ""), re.I)]
print("Cot nhay cam bi bo:", sens)
if sens:
    tr_ = df[df.WEEK_NUM <= 63]; va_ = df[(df.WEEK_NUM >= 64) & (df.WEEK_NUM <= 77)]; te_ = df[df.WEEK_NUM >= 78]
    fs = [c for c in feats if c not in sens]
    P_MAIN = dict(P_FAST, learning_rate=0.03, n_estimators=3000)
    m_s = lgb.LGBMClassifier(**P_MAIN).fit(tr_[fs], tr_["target"], eval_set=[(va_[fs], va_["target"])],
                                           callbacks=[lgb.early_stopping(200, verbose=False)])
    p_s = m_s.predict_proba(te_[fs])[:, 1]
    p_f = (fpack["model"].predict_proba(te_[feats])[:, 1] if HAS_FINAL else pt.set_index("case_id").loc[te_["case_id"], best].values)
    r = utils.delong_test(te_["target"], p_f, p_s)
    print("AUC co gioi tinh %.4f | khong gioi tinh %.4f | chenh %.4f | p = %.3g" % (r["AUC_1"], r["AUC_2"], r["chenh"], r["p_value"]))
    summary["AUC bo gioi tinh (chenh, p)"] = "%.4f (%.4f, p=%.2g)" % (r["AUC_2"], r["chenh"], r["p_value"])
    g = te_[sens[0]].astype(str)
    for name, p in [("co gioi tinh", p_f), ("khong gioi tinh", p_s)]:
        thr = np.quantile(p, 0.90)                               # duyet 90% ho so rui ro thap nhat
        ok = pd.Series(p <= thr, index=te_.index)
        t = pd.DataFrame({"ty_le_duyet": ok.groupby(g).mean(), "vo_no_nhom_duyet": te_["target"][ok].groupby(g[ok]).mean()})
        print("\n", name); display(t.round(4))

## 7. Tổng hợp kiểm định

In [ ]:
s = pd.Series(summary, name="gia_tri").to_frame(); s.to_csv(OUT_DIR / "validation_summary.csv"); s

## 8. Nhận xét (tự viết)
- Độ chính xác: AUC/Gini và khoảng tin cậy; chênh lệch với mô hình nền có ý nghĩa thống kê không (DeLong): ...
- Độ ổn định: Gini stability, kết quả cửa sổ mở rộng vs cửa sổ trượt (có nên huấn luyện lại định kỳ không): ...
- Hiệu chỉnh: PD sau isotonic có dùng được cho tính tổn thất kỳ vọng / hạn mức không: ...
- Trôi dữ liệu và rò rỉ: AUC adversarial, các biến trôi mạnh cần giám sát; kết quả hoán vị nhãn: ...
- Kết luận về khả năng ứng dụng: ...